# Data Preparation & Feature Engineering
This notebook covers data loading, exploration, location/user/temporal feature extraction, and data cleanup for tourism reviews.

In [ ]:
import pandas as pd
import re
from pathlib import Path

INPUT_CSV = Path("../data/raw/Reviews.csv")
if not INPUT_CSV.exists():
    raise FileNotFoundError(f"Input file not found: {INPUT_CSV.resolve()}")

df = pd.read_csv(INPUT_CSV, encoding='latin1')

# Basic inspection
print("Dataset Overview:")
print(f"  Shape: {df.shape}")
print(f"  Columns: {df.columns.tolist()}")
print(f"\n Data Types:")
print(df.dtypes)
print(f"\n Missing Values:")
print(df.isnull().sum())
print(f"\n Duplicates: {df.duplicated().sum()}")
print(f"\n Rating Distribution:")
print(df["Rating"].value_counts().sort_index())
print(f"\n Top 10 Cities:")
print(df["Located_City"].value_counts().head(10))

In [ ]:


provinces = [
    "Western Province", "Central Province", "Southern Province",
    "Northern Province", "Eastern Province", "North Western Province",
    "North Central Province", "Uva Province", "Sabaragamuwa Province",
]

city_to_district = {
    "Arugam Bay": "Ampara",
    "Ampara": "Ampara",
    "Galle": "Galle",
    "Kalutara": "Kalutara",
    "Trincomalee": "Trincomalee",
    "Matara": "Matara",
    "Colombo": "Colombo",
    "Gampaha": "Gampaha",
    "Nilaveli": "Trincomalee",
    "Kalkudah": "Batticaloa",
    "Nuwara Eliya": "Nuwara Eliya",
    "Kandy": "Kandy",
    "Tissamaharama": "Hambantota",
    "Anuradhapura": "Anuradhapura",
    "Haputale": "Badulla",
    "Badulla": "Badulla",
    "Beruwala": "Kalutara",
    "Jaffna": "Jaffna",
    "Polonnaruwa": "Polonnaruwa",
    "Matale": "Matale",
    "Weligatta": "Hambantota",
    "Ratnapura": "Ratnapura",
    "Tangalle": "Hambantota",
    "Pinnawala": "Kegalle",
    "Deniyaya": "Matara",
    "Koslanda": "Badulla",
    "Mirissa": "Matara",
    "Ella": "Badulla",
    "Negombo": "Gampaha",
    "Sigiriya": "Matale",
    "Batticaloa": "Batticaloa",
    "Bentota": "Galle",
    "Hikkaduwa": "Galle",
    "Mirigama": "Gampaha",
    "Habarana": "Anuradhapura",
}

manual_location_mappings = {
    "Udawalawe National Park": {"province": "Sabaragamuwa Province", "district": "Ratnapura"},
    "North Central Province": {"province": "North Central Province", "district": "Anuradhapura"},
}

province_pattern = re.compile(
    r"(" + "|".join([re.escape(p) for p in provinces]) + r")", flags=re.IGNORECASE
)

def extract_province(location_str):
    """Extract province from location string"""
    if not isinstance(location_str, str) or not location_str.strip():
        return None
    
    loc = location_str.strip()
    
    # Manual mapping
    for k, v in manual_location_mappings.items():
        if k.lower() == loc.lower():
            return v.get("province")
    
    # Pattern matching
    m = province_pattern.search(loc)
    if m:
        return m.group(1)
    
    # Numeric pattern
    m2 = re.search(r"([A-Za-z ]+Province)\b", loc)
    if m2:
        return m2.group(1).strip()
    
    return None

def infer_district(row):
    """Infer district from location and city"""
    city = row.get("Located_City")
    location = row.get("Location")
    
    # Manual mapping
    if isinstance(location, str):
        for k, v in manual_location_mappings.items():
            if k.lower() == location.strip().lower():
                return v.get("district")
    
    # City mapping
    if isinstance(city, str) and city in city_to_district:
        return city_to_district[city]
    
    if not isinstance(location, str):
        return None
    
    parts = [p.strip() for p in location.split(",") if p.strip()]
    
    # Check for explicit district
    for part in parts:
        if "District" in part:
            return part.replace("District", "").strip()
    
    # Infer from position
    if len(parts) >= 3:
        return parts[-2]
    elif len(parts) == 2:
        return parts[0]
    elif len(parts) == 1 and not any(parts[0].lower() == p.lower() for p in provinces):
        return parts[0]
    
    return None

# Apply location extraction
print(" Extracting province and district...")
df["Province"] = df["Location"].apply(extract_province)
df["District"] = df.apply(infer_district, axis=1)

print(f"Province coverage: {df['Province'].notna().sum()}/{len(df)} rows")
print(f"District coverage: {df['District'].notna().sum()}/{len(df)} rows")
print(f"\n Top Provinces:")
print(df["Province"].value_counts().head(10))

In [ ]:
import re
from pathlib import Path

import pycountry
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter
from geopy.exc import GeocoderTimedOut, GeocoderServiceError

# Reproducibility controls: offline by default, optional network geocoding.
ENABLE_GEOCODING = False
CACHE_PATH = Path("data/cache/country_resolution_cache.csv")
CACHE_PATH.parent.mkdir(parents=True, exist_ok=True)
SAVE_CACHE_UPDATES = True

def normalize_location(value):
    """Normalize raw location text into a consistent format."""
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return None

    text = str(value).strip().lower()
    if not text:
        return None

    text = re.sub(r"[^a-z0-9,\s\-.]", " ", text)
    text = re.sub(r"\s*,\s*", ", ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip(" ,")

def build_country_lookup():
    """Build alias -> canonical country map using pycountry + common aliases."""
    alias_map = {}

    for country in pycountry.countries:
        canonical = country.name
        aliases = {country.name, country.alpha_2, country.alpha_3}

        if hasattr(country, "official_name"):
            aliases.add(country.official_name)
        if hasattr(country, "common_name"):
            aliases.add(country.common_name)

        for alias in aliases:
            key = normalize_location(alias)
            if key:
                alias_map[key] = canonical

    manual_aliases = {
        "usa": "United States",
        "u.s.a": "United States",
        "us": "United States",
        "u.s": "United States",
        "united states of america": "United States",
        "uk": "United Kingdom",
        "u.k": "United Kingdom",
        "uae": "United Arab Emirates",
        "u.a.e": "United Arab Emirates",
        "england": "United Kingdom",
        "scotland": "United Kingdom",
        "wales": "United Kingdom",
        "northern ireland": "United Kingdom",
        "sri lanka": "Sri Lanka",
    }

    for alias, canonical in manual_aliases.items():
        alias_map[normalize_location(alias)] = canonical

    return alias_map

COUNTRY_LOOKUP = build_country_lookup()
COUNTRY_ALIASES_SORTED = sorted(COUNTRY_LOOKUP.keys(), key=len, reverse=True)

def extract_country_direct(normalized_text):
    """Try direct country extraction from normalized location string."""
    if not normalized_text:
        return None

    for alias in COUNTRY_ALIASES_SORTED:
        pattern = rf"(^|[^a-z0-9]){re.escape(alias)}([^a-z0-9]|$)"
        if re.search(pattern, normalized_text):
            return COUNTRY_LOOKUP[alias]
    return None

def infer_country_from_locale(locale_value):
    """Infer country name from locale like en_US, en-GB, ru_RU."""
    if not isinstance(locale_value, str) or not locale_value.strip():
        return None

    locale = locale_value.strip().replace("-", "_")
    parts = [p for p in locale.split("_") if p]
    if len(parts) < 2:
        return None

    cc = parts[-1].upper()
    special = {
        "UK": "GB",
        "EL": "GR",
        "EN": None,
    }
    cc = special.get(cc, cc)
    if cc is None or len(cc) != 2:
        return None

    country = pycountry.countries.get(alpha_2=cc)
    return country.name if country else None

def load_cache(path):
    if not path.exists():
        return {}
    cache_df = pd.read_csv(path)
    required = {"normalized_location", "country", "status"}
    if not required.issubset(cache_df.columns):
        raise ValueError(f"Cache file must contain columns: {required}")
    cache = {}
    for _, row in cache_df.dropna(subset=["normalized_location"]).iterrows():
        key = str(row["normalized_location"]).strip()
        country_raw = row["country"] if pd.notna(row["country"]) else None
        country = None if country_raw in (None, "", "Unknown") else country_raw
        status = row["status"] if pd.notna(row["status"]) else "cached_unknown"
        cache[key] = {"country": country, "status": status}
    return cache

def save_cache(path, cache):
    rows = []
    for loc, info in cache.items():
        if loc is None:
            continue
        country_value = info.get("country") if info.get("country") else "Unknown"
        rows.append({
            "normalized_location": loc,
            "country": country_value,
            "status": info.get("status", "unknown"),
        })
    cache_out = pd.DataFrame(rows).sort_values("normalized_location")
    cache_out.to_csv(path, index=False)

# Optional geocoder (disabled by default for deterministic offline runs).
if ENABLE_GEOCODING:
    geolocator = Nominatim(user_agent="tourism_country_resolution_v2")
    geocode = RateLimiter(
        geolocator.geocode,
        min_delay_seconds=1.0,
        max_retries=2,
        error_wait_seconds=2.0,
        swallow_exceptions=True,
    )
else:
    geocode = None

def geocode_country(normalized_text):
    if not ENABLE_GEOCODING or geocode is None:
        return None
    try:
        result = geocode(normalized_text, addressdetails=True, language="en")
        if result and isinstance(result.raw, dict):
            address = result.raw.get("address", {})
            country = address.get("country")
            if country:
                return country
    except (GeocoderTimedOut, GeocoderServiceError, Exception):
        return None
    return None

existing_cache = load_cache(CACHE_PATH)
cache_updated = False

# Parse each unique normalized location once (cache).
df["_normalized_user_location"] = df["User_Location"].apply(normalize_location)
unique_locations = df["_normalized_user_location"].drop_duplicates().tolist()
resolution_cache = dict(existing_cache)

for loc in unique_locations:
    if loc is None:
        resolution_cache[loc] = {"country": None, "status": "missing"}
        continue

    if loc in resolution_cache:
        continue

    direct_country = extract_country_direct(loc)
    if direct_country is not None:
        resolution_cache[loc] = {"country": direct_country, "status": "direct_match"}
        cache_updated = True
        continue

    geo_country = geocode_country(loc)
    if geo_country is not None:
        resolution_cache[loc] = {"country": geo_country, "status": "geocoded"}
        cache_updated = True
    else:
        status = "not_found_offline" if not ENABLE_GEOCODING else "not_found"
        resolution_cache[loc] = {"country": None, "status": status}
        cache_updated = True

if SAVE_CACHE_UPDATES and cache_updated:
    save_cache(CACHE_PATH, resolution_cache)

# Map resolved results back to full dataframe.
df["User_Country"] = df["_normalized_user_location"].map(lambda x: resolution_cache.get(x, {}).get("country"))
df["resolution_status"] = df["_normalized_user_location"].map(lambda x: resolution_cache.get(x, {}).get("status", "missing"))

# Fallback 1: infer from locale when location-based country is unresolved.
locale_fallback_mask = df["User_Country"].isna()
if locale_fallback_mask.any():
    inferred = df.loc[locale_fallback_mask, "User_Locale"].apply(infer_country_from_locale)
    fill_idx = inferred[inferred.notna()].index
    df.loc[fill_idx, "User_Country"] = inferred.loc[fill_idx]
    df.loc[fill_idx, "resolution_status"] = "locale_fallback"

# Fallback 2: force full completeness with explicit placeholder.
unknown_mask = df["User_Country"].isna()
if unknown_mask.any():
    df.loc[unknown_mask, "User_Country"] = "Unknown"
    df.loc[unknown_mask, "resolution_status"] = "unknown_fallback"

df["User_Region"] = None

print("Parsing user locations with direct matching + cached resolution...")
print(f"Geocoding enabled: {ENABLE_GEOCODING}")
print(f"Country resolution: {df['User_Country'].notna().sum()}/{len(df)} rows")
print("\nResolution status counts:")
print(df["resolution_status"].value_counts(dropna=False))
print("\nTop 15 Countries:")
print(df["User_Country"].value_counts().head(15))
print(f"\nRows still missing User_Country: {df['User_Country'].isna().sum()}")

# Drop helper column used for caching keys.
df = df.drop(columns=["_normalized_user_location"], errors="ignore")

In [ ]:
# Convert dates
for col in ['Travel_Date', 'Published_Date']:
    df[col] = pd.to_datetime(df[col], errors='coerce', utc=True)
    df[f'{col}_Month'] = df[col].dt.month
    df[f'{col}_Year'] = df[col].dt.year

# Add text features
df['Review_Length'] = df['Text'].apply(len)
df['Title_Length'] = df['Title'].apply(len)

print("Temporal features extracted")
print(f"\n Date Range:")
print(f"  Travel: {df['Travel_Date'].min()} to {df['Travel_Date'].max()}")
print(f"  Published: {df['Published_Date'].min()} to {df['Published_Date'].max()}")

In [ ]:
# --- Add Review Delay and Rating Class Columns ---
def get_rating_class(r):
    if r <= 2:
        return "Negative"
    elif r == 3:
        return "Neutral"
    else:
        return "Positive"

df["Rating_Class"] = df["Rating"].apply(get_rating_class)

if "Published_Date" in df.columns and "Travel_Date" in df.columns:
    df["Review_Delay_Days"] = (df["Published_Date"] - df["Travel_Date"]).dt.days
    # Fix negative values (bad data)
    df["Review_Delay_Days"] = df["Review_Delay_Days"].clip(lower=0)
else:
    df["Review_Delay_Days"] = None

print("Added Review_Delay_Days and Rating_Class columns (negative delays fixed)")

In [ ]:
# Drop raw location columns (redundant with extracted features)
df = df.drop(columns=['Location', 'User_Location', 'User_Region'], errors='ignore')

# Drop raw date columns after temporal features are created
for col in ['Travel_Date', 'Published_Date']:
    if col in df.columns:
        df = df.drop(col, axis=1)
        print(f"'{col}' column dropped.")
    else:
        print(f"'{col}' column not found.")

# Drop User_ID column if it exists
if 'User_ID' in df.columns:
    df = df.drop('User_ID', axis=1)
    print("'User_ID' column dropped.")
else:
    print("'User_ID' column not found.")

# Fail fast if required columns are missing before export
required_cols = [
    'Location_Name', 'Located_City', 'Province', 'District', 'Location_Type',
    'User_Locale', 'User_Country', 'Travel_Date_Month', 'Travel_Date_Year',
    'Published_Date_Month', 'Published_Date_Year',
    'User_Contributions', 'Rating', 'Helpful_Votes',
    'Title', 'Text', 'Review_Length', 'Title_Length', 'Rating_Class', 'Review_Delay_Days'
    ]
missing_required = [c for c in required_cols if c not in df.columns]
if missing_required:
    raise ValueError(f"Missing required columns before export: {missing_required}")

# Reorder columns
ordered_cols = required_cols.copy()
ordered_cols = [col for col in ordered_cols if col in df.columns]
df = df[ordered_cols + [col for col in df.columns if col not in ordered_cols]]

# Keep output schema stable by removing operational helper columns
df.drop(columns=['resolution_status'], inplace=True, errors='ignore')

print("Cleaned up redundant columns and reordered")
print(f"\nDataset Shape: {df.shape}")
print(f"\nCurrent Columns ({len(df.columns)}):")
for i, col in enumerate(df.columns, 1):
    print(f"  {i:2d}. {col}")

# Timestamped output avoids accidental overwrite of previous runs
# run_tag = pd.Timestamp.utcnow().strftime('%Y%m%d_%H%M%S')
# output_file = f"Processed_Reviews_dataset_prep_{run_tag}.csv"
output_file = "data/processed/Processed_Reviews.csv"
Path(output_file).parent.mkdir(parents=True, exist_ok=True)
df.to_csv(output_file, index=False)
print(f"\nSaved reproducible output file: {output_file}")

In [ ]:
df["District"].unique().tolist()

In [ ]:
df.head()